# Residual Signal Finder on UCI credit data

This notebook explores the public functionality of `ResidualSignalFinder` on a real UCI Machine Learning Repository dataset: **Default of Credit Card Clients**. The dataset is credit-related, has 30,000 observations, and provides 23 modeling features after dropping the ID column.

Workflow:

1. Load the saved UCI dataset from `data/default_of_credit_card_clients.csv`.
2. Fit a simple GLM-style baseline model using all features to predict default probability.
3. Pass those baseline predictions into `ResidualSignalFinder` as `y_pred`.
4. Inspect CV, holdout, group CV, bootstrap, interactions, plots, reports, and v1 validation behavior.


## Setup

The path setup lets the notebook run from the repository root or from inside `notebooks/`.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

for path in [PROJECT_ROOT / 'src', PROJECT_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

PROJECT_ROOT


WindowsPath('C:/Users/amsac/Desktop/repos/modeling-tools')

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from pe_tools.signal_finder import ResidualSignalFinder

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)


## Load the UCI credit dataset

The target is `default_next_month`. All remaining columns are numeric features, including several numerically encoded categorical fields (`sex`, `education`, and `marriage`). For v1 of the residual finder, this is acceptable because the implementation explicitly accepts numeric columns only.


In [3]:
DATA_PATH = PROJECT_ROOT / 'data' / 'default_of_credit_card_clients.csv'
credit = pd.read_csv(DATA_PATH)
target_col = 'default_next_month'
feature_cols = [column for column in credit.columns if column != target_col]

X_all = credit.loc[:, feature_cols].copy()
y_all = credit[target_col].astype(float).rename(target_col)

print(f'Rows: {len(credit):,}')
print(f'Feature count: {len(feature_cols)}')
display(X_all.head())
display(y_all.value_counts(normalize=True).rename('target_rate'))


Rows: 30,000
Feature count: 23


,limit_bal,sex,education,marriage,age,pay_0,pay_2,pay_3,pay_4,pay_5,pay_6,bill_amt1,bill_amt2,bill_amt3,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6
0,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0
1,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000
2,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000
3,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000
4,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679


default_next_month
0.0    0.7788
1.0    0.2212
Name: target_rate, dtype: float64

## Create a fast modeling sample and evaluation splits

The full CSV remains available in `data/`, but this notebook uses a stratified 6,000-row sample so the full feature tour runs quickly. The explicit split labels are reused for holdout mode.


In [4]:
analysis_frame = train_test_split(
    credit,
    train_size=6_000,
    stratify=credit[target_col],
    random_state=42,
)[0].sort_index()

train_valid, holdout_rows = train_test_split(
    analysis_frame,
    test_size=0.15,
    stratify=analysis_frame[target_col],
    random_state=42,
)
train_rows, validation_rows = train_test_split(
    train_valid,
    test_size=0.1765,
    stratify=train_valid[target_col],
    random_state=42,
)

X = analysis_frame.loc[:, feature_cols].copy()
y_true = analysis_frame[target_col].astype(float).rename(target_col)
split_col = pd.Series('train', index=X.index, name='split')
split_col.loc[validation_rows.index] = 'validation'
split_col.loc[holdout_rows.index] = 'holdout'

display(split_col.value_counts().rename('row_count'))


split
train         4199
validation     901
holdout        900
Name: row_count, dtype: int64

## Build sample weights

Weights are optional, but they are useful here to balance the minority default class while slightly emphasizing larger credit limits.


In [5]:
default_rate = float(y_true.mean())
class_weight = np.where(
    y_true.eq(1),
    0.5 / default_rate,
    0.5 / (1.0 - default_rate),
)
limit_weight = (X['limit_bal'] / X['limit_bal'].median()).clip(0.25, 4.0)
sample_weight = pd.Series(
    class_weight * limit_weight.to_numpy(),
    index=X.index,
    name='sample_weight',
)

display(sample_weight.describe())


count    6000.000000
mean        1.114565
std         1.143140
min         0.160496
25%         0.366849
50%         0.807407
75%         1.375684
max         9.042954
Name: sample_weight, dtype: float64

## Fit a simple GLM baseline

The baseline is intentionally plain: standardized features plus logistic regression. It is GLM-like, uses all modeling features, and produces default probabilities. Those probabilities become the residual finder's `y_pred`.


In [6]:
train_mask = split_col.eq('train')
eval_mask = ~train_mask
baseline_glm = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1_000, solver='lbfgs'),
)
baseline_glm.fit(
    X.loc[train_mask],
    y_true.loc[train_mask],
    logisticregression__sample_weight=sample_weight.loc[train_mask],
)

glm_y_pred = pd.Series(
    baseline_glm.predict_proba(X)[:, 1],
    index=X.index,
    name='glm_default_probability',
)
holdout_auc = roc_auc_score(y_true.loc[eval_mask], glm_y_pred.loc[eval_mask])
print(f'GLM validation plus holdout ROC AUC: {holdout_auc:.3f}')
display(
    pd.DataFrame({'y_true': y_true, 'glm_y_pred': glm_y_pred})
    .assign(residual=lambda frame: frame['y_true'] - frame['glm_y_pred'])
    .head()
)


GLM validation plus holdout ROC AUC: 0.719


,y_true,glm_y_pred,residual
5,0.0,0.570879,-0.570879
13,1.0,0.545202,0.454798
24,0.0,0.493582,-0.493582
40,0.0,0.002966,-0.002966
41,0.0,0.513133,-0.513133


## Default CV residual scan

The default split strategy is shuffled K-fold CV. Feature importance is gain-based and stability summarizes how often each feature ranks near the top across folds.


In [ ]:
cv_result = ResidualSignalFinder(
    model_type='xgboost',
    max_depth=1,
    n_estimators=80,
    learning_rate=0.05,
    n_bins=10,
    split_strategy='cv',
    n_splits=5,
    random_state=42,
).fit(
    X=X,
    y_true=y_true,
    y_pred=glm_y_pred,
    sample_weight=sample_weight,
)

print(cv_result.metadata)
display(cv_result.residual_model_score)
display(cv_result.fold_scores)
display(cv_result.feature_importance.head(12))
display(cv_result.feature_stability.head(12))


In [ ]:
pd.testing.assert_series_equal(cv_result.residuals, y_true - glm_y_pred, check_names=False)
cv_result.residuals.describe()


## Stratified CV scan

When `stratify_col` is provided, the finder bins that numeric column and uses `StratifiedKFold`. This can be handy when a feature such as credit limit should be balanced across folds.


In [ ]:
stratified_result = ResidualSignalFinder(
    model_type='xgboost',
    max_depth=1,
    n_estimators=40,
    learning_rate=0.05,
    split_strategy='cv',
    stratify_col='limit_bal',
    n_splits=4,
    random_state=42,
).fit(
    X=X,
    y_true=y_true,
    y_pred=glm_y_pred,
    sample_weight=sample_weight,
)

display(stratified_result.fold_scores)
display(stratified_result.feature_importance.head(8))


## Binned diagnostics

Each feature gets a binned diagnostics table using held-out fold predictions where available. These tables power the plotting and report helpers.


In [ ]:
top_feature = cv_result.feature_importance.iloc[0]['feature']
print(f'Top feature: {top_feature}')
display(cv_result.binned_diagnostics[top_feature])


## Plot top residual signals

`plot_top_residual_signals` returns Matplotlib `Figure` objects. It only saves files when `save_dir` is provided.


In [ ]:
figures = cv_result.plot_top_residual_signals(top_n=3)
for feature, figure in figures.items():
    print(feature, type(figure).__name__)
    display(figure)
    plt.close(figure)


## Holdout scan

Holdout mode uses explicit `train`, `validation`, and `holdout` labels. The residual model is fit on `train` and scored on each available evaluation split.


In [ ]:
holdout_result = ResidualSignalFinder(
    model_type='xgboost',
    max_depth=1,
    n_estimators=80,
    learning_rate=0.05,
    split_strategy='holdout',
    random_state=42,
).fit(
    X=X,
    y_true=y_true,
    y_pred=glm_y_pred,
    sample_weight=sample_weight,
    split_col=split_col,
)

display(holdout_result.fold_scores)
display(holdout_result.residual_model_score)
display(holdout_result.feature_importance.head(12))


## Group CV scan

Group CV requires a numeric grouping column inside `X` for v1. This example groups by age decile, and the finder drops the group column from model features before fitting.


In [ ]:
age_group = pd.qcut(X['age'], q=5, labels=False, duplicates='drop').astype(int)
X_grouped = X.assign(age_group=age_group)
group_result = ResidualSignalFinder(
    model_type='random_forest',
    max_depth=2,
    n_estimators=60,
    split_strategy='group_cv',
    group_col='age_group',
    n_splits=5,
    random_state=42,
).fit(
    X=X_grouped,
    y_true=y_true,
    y_pred=glm_y_pred,
    sample_weight=sample_weight,
)

group_columns = ['fold', 'test_groups', 'train_size', 'test_size', 'test_r2']
display(group_result.fold_scores[group_columns])
display(group_result.feature_importance.head(12))
group_feature_excluded = 'age_group' not in set(group_result.feature_importance['feature'])
print('Group feature excluded from model:', group_feature_excluded)


## Bootstrap scan

Bootstrap mode repeatedly samples training rows and scores out-of-bag rows when available. This gives another view of feature stability.


In [ ]:
bootstrap_result = ResidualSignalFinder(
    model_type='xgboost',
    max_depth=1,
    n_estimators=60,
    learning_rate=0.05,
    split_strategy='bootstrap',
    n_repeats=6,
    sample_fraction=0.8,
    random_state=42,
).fit(
    X=X,
    y_true=y_true,
    y_pred=glm_y_pred,
    sample_weight=sample_weight,
)

display(bootstrap_result.fold_scores)
display(bootstrap_result.feature_stability.head(12))


## Interaction scan with `max_depth=2`

When `max_depth=2`, the finder attempts conservative SHAP interaction detection if `shap` is installed. If SHAP is unavailable, metadata records a warning and `interaction_importance` remains empty.


In [ ]:
interaction_result = ResidualSignalFinder(
    model_type='xgboost',
    max_depth=2,
    n_estimators=80,
    learning_rate=0.05,
    split_strategy='cv',
    n_splits=3,
    random_state=42,
).fit(
    X=X,
    y_true=y_true,
    y_pred=glm_y_pred,
    sample_weight=sample_weight,
)

display(interaction_result.metadata.get('warnings', []))
display(interaction_result.interaction_importance.head(12))


## Report export

Results can be exported to Excel and HTML. The notebook writes reports into `notebooks/outputs/`.


In [ ]:
report_dir = PROJECT_ROOT / 'notebooks' / 'outputs'
report_dir.mkdir(parents=True, exist_ok=True)

excel_path = cv_result.to_excel(report_dir / 'uci_credit_residual_signal_report.xlsx')
html_path = cv_result.to_html(report_dir / 'uci_credit_residual_signal_report.html')

print(f'Wrote {excel_path.relative_to(PROJECT_ROOT)}')
print(f'Wrote {html_path.relative_to(PROJECT_ROOT)}')


## v1 validation behavior

The current v1 finder intentionally rejects non-numeric feature columns and missing values. The original UCI fields are numeric, so the examples below inject bad inputs to demonstrate the guardrails.


In [ ]:
try:
    ResidualSignalFinder(random_state=42).fit(
        X=X.assign(risk_bucket=np.where(y_true.eq(1), 'default', 'current')),
        y_true=y_true,
        y_pred=glm_y_pred,
    )
except ValueError as error:
    print(f'Categorical rejection: {error}')

try:
    X_missing = X.copy()
    X_missing.loc[X_missing.index[0], 'limit_bal'] = pd.NA
    ResidualSignalFinder(random_state=42).fit(
        X=X_missing,
        y_true=y_true,
        y_pred=glm_y_pred,
    )
except ValueError as error:
    print(f'Missing-value rejection: {error}')
